In [2]:
import pandas as pd
import pandas as pd
import re
import numpy as np

In [3]:
data = pd.read_csv("combined_temperature.csv")

In [4]:
data 

,Country,Year,Annual Mean,5-yr smooth,Code
0,Dominican Republic,1901,23.71,23.65,DOM
1,Dominican Republic,1902,23.40,23.52,DOM
2,Dominican Republic,1903,23.45,23.38,DOM
3,Dominican Republic,1904,23.03,23.26,DOM
4,Dominican Republic,1905,23.44,23.16,DOM
...,...,...,...,...,...
20735,Switzerland,2018,7.68,7.27,CHE
20736,Switzerland,2019,7.31,7.34,CHE
20737,Switzerland,2020,7.64,7.38,CHE
20738,Switzerland,2021,6.39,7.42,CHE


In [5]:
print(data.dtypes)

Country         object
Year             int64
Annual Mean    float64
5-yr smooth    float64
Code            object
dtype: object


In [6]:
data = data.rename(columns={
    "Annual Mean": "Annual_Mean",
    "5-yr smooth": "Five_Year_Smooth"
})

In [7]:
data["Year"] = pd.to_datetime(data["Year"], format="%Y")

In [8]:
data  

,Country,Year,Annual_Mean,Five_Year_Smooth,Code
0,Dominican Republic,1901-01-01,23.71,23.65,DOM
1,Dominican Republic,1902-01-01,23.40,23.52,DOM
2,Dominican Republic,1903-01-01,23.45,23.38,DOM
3,Dominican Republic,1904-01-01,23.03,23.26,DOM
4,Dominican Republic,1905-01-01,23.44,23.16,DOM
...,...,...,...,...,...
20735,Switzerland,2018-01-01,7.68,7.27,CHE
20736,Switzerland,2019-01-01,7.31,7.34,CHE
20737,Switzerland,2020-01-01,7.64,7.38,CHE
20738,Switzerland,2021-01-01,6.39,7.42,CHE


In [9]:
data.to_csv("cleaned_temperature.csv", index=False)

In [10]:
energy = pd.read_csv("energy_cleaned.csv")
temperature = pd.read_csv("temperature_cleaned.csv")
merged = pd.read_csv("merged_final.csv")

print("Loaded successfully")
print(energy.columns.tolist())

Loaded successfully
['Country', 'Energy Type', 'Year', 'Energy Consumption', 'Energy Production', 'GDP', 'Population', 'Energy Intensity Per Capita', 'Energy Intensity By GDP', 'CO2 Emission']


In [11]:
# Standardize column names after loading
energy.columns = energy.columns.str.strip().str.replace(" ", "_").str.lower()
temperature.columns = temperature.columns.str.strip().str.replace(" ", "_").str.lower()
merged.columns = merged.columns.str.strip().str.replace(" ", "_").str.lower()

# Verify
print(energy.columns.tolist())

['country', 'energy_type', 'year', 'energy_consumption', 'energy_production', 'gdp', 'population', 'energy_intensity_per_capita', 'energy_intensity_by_gdp', 'co2_emission']


In [12]:
print(energy[energy["country"].str.contains("chin", case=False)]["country"].unique())

['China']


In [13]:
print(merged[merged["country"].str.contains("chin", case=False)]["country"].unique())

[]


In [15]:
print(temperature[temperature["country"].str.contains("chin", case=False)]["country"].unique())

[]


In [16]:
print(energy[energy["country"].str.contains("chin", case=False)]["country"].unique())

['China']


In [27]:
energy_countries = set(energy["country"].unique())
temp_countries = set(temperature["country"].unique())

print("In energy but not temperature:")
print(sorted(energy_countries - temp_countries))


print("\nIn temperature but not energy:")
print(sorted(temp_countries - energy_countries))

In energy but not temperature:
['American Samoa', 'Antarctica', 'Antigua and Barbuda', 'Aruba', 'Bermuda', 'Bosnia and Herzegovina', 'British Virgin Islands', 'Brunei', 'Burkina Faso', 'Burma', 'Cabo Verde', 'Cayman Islands', 'China', 'Congo-Brazzaville', 'Congo-Kinshasa', 'Cook Islands', 'Czech Republic', 'Côte d’Ivoire', 'El Salvador', 'Equatorial Guinea', 'Falkland Islands', 'Faroe Islands', 'Former Czechoslovakia', 'Former Serbia and Montenegro', 'Former U.S.S.R.', 'Former Yugoslavia', 'French Guiana', 'French Polynesia', 'Gambia, The', 'Germany, East', 'Germany, West', 'Gibraltar', 'Greenland', 'Guadeloupe', 'Guam', 'Guinea-Bissau', 'Hawaiian Trade Zone', 'Hong Kong', 'India', 'Iran', 'Ireland', 'Kosovo', 'Kyrgyzstan', 'Laos', 'Lithuania', 'Macau', 'Martinique', 'Micronesia', 'Montserrat', 'Morocco', 'Namibia', 'Netherlands Antilles', 'New Caledonia', 'Niue', 'North Korea', 'North Macedonia', 'Northern Mariana Islands', 'Palestinian Territories', 'Puerto Rico', 'Reunion', 'Russia'

In [34]:
energy["country"] = energy["country"].replace({
    "Côte d'Ivoire": "Côte d'Ivoire",  # encoding fix
    "Congo-Brazzaville": "Congo-Kinshasa",  # closest match
    "Burma": "Burma",  # already matches
    "Gambia, The": "Gambia, The", })

In [35]:
merged = pd.merge(
    energy[energy["energy_type"] == "all_energy_types"],
    temperature,
    on=["country", "year"],
    how="left"
)

In [36]:
print(f"Countries in merged: {merged['country'].nunique()}")


Countries in merged: 229


In [37]:
for country in ["China", "India", "United States", "Russia"]:
    rows = len(merged[merged["country"] == country])
    has_temp = merged[merged["country"] == country]["annual_mean"].notna().sum()
    print(f"{country}: {rows} rows, {has_temp} with temperature data")

China: 40 rows, 0 with temperature data
India: 40 rows, 0 with temperature data
United States: 40 rows, 40 with temperature data
Russia: 40 rows, 40 with temperature data


In [38]:
merged.to_csv("merged_final.csv", index=False)